# Phase 2 - Hyperparameter tuning (Colab T4)

Tunes `lr0`, `box`, `cls`, `dfl` for the **hazard** detector (the one with
headroom: baseline mAP50 = 0.566 vs child's 0.947).

**Budget:** 6 iterations x 20 epochs is about **2.9 h** on a T4. Raise
`--iterations` only if you have session time to spare - see the table in the
tuning cell.

**Resumable:** results are appended after every iteration and re-read on
restart, so a disconnect costs at most the in-flight iteration. Just re-run
the same cell to continue the sweep.

Setup steps are identical to `colab_train.ipynb`: T4 runtime + the
`ROBOFLOW_API_KEY` Colab Secret.

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q ultralytics roboflow

In [ ]:
import os
REPO_URL = "https://github.com/FooJames/DEEPLRN_Group2.git"
if not os.path.isdir("DEEPLRN_Group2"):
    !git clone $REPO_URL
else:
    !cd DEEPLRN_Group2 && git pull
%cd DEEPLRN_Group2

In [ ]:
from google.colab import userdata
import os
os.environ["ROBOFLOW_API_KEY"] = userdata.get("ROBOFLOW_API_KEY")
print("key loaded:", bool(os.environ.get("ROBOFLOW_API_KEY")))

In [ ]:
# Hazard dataset only (child isn't being tuned in this pass)
!python scripts/download_data.py --child-version 3 --hazard-version 1 --only hazard
!python scripts/fix_data_yaml.py data/hazard/data.yaml

## Tune the hazard detector

T4 budget (measured: 1.45 min/epoch on this dataset):

| iterations | 15 ep | 20 ep | 30 ep |
|---|---|---|---|
| 6  | 2.2 h | **2.9 h** | 4.3 h |
| 8  | 2.9 h | 3.9 h | 5.8 h |
| 10 | 3.6 h | 4.8 h | 7.2 h |

20 epochs is a *ranking* proxy, not convergence (hazard reaches ~0.46 mAP50
by epoch 20 vs 0.53 at 100) - fine for comparing configs, and the winner is
retrained at full length in Phase 4.

`--optimizer AdamW` is deliberate: `auto` silently ignores `lr0`.

In [ ]:
!python scripts/tune.py --model hazard --data data/hazard/data.yaml     --iterations 6 --epochs 20 --optimizer AdamW

### Save the sweep (run right after tuning)

In [ ]:
# The whole sweep + the winning config. Verify weights-free bundle lists both files.
!zip -r hazard_tuning.zip runs/detect/tune_hazard results/metrics/tuning_hazard.csv results/metrics/tuning_hazard_best.yaml
!unzip -l hazard_tuning.zip
from google.colab import files
files.download("hazard_tuning.zip")

### Results

In [ ]:
# Every iteration, best first (fitness = weighted mAP)
import pandas as pd
df = pd.read_csv("results/metrics/tuning_hazard.csv")
print(df.sort_values("fitness", ascending=False).to_string(index=False))
print("
baseline fitness to beat: hazard mAP50=0.5657 / mAP50-95=0.4072")

In [ ]:
!cat results/metrics/tuning_hazard_best.yaml

### Notes
- Tuning evaluates on the **val** split only; the test split stays untouched
  until the very end.
- Child tuning is intentionally skipped (0.947 baseline = little headroom). If
  you do run it later, it's the same command with `--model child`.
- Caveat for the write-up: `lr0` is tuned under AdamW, so the Phase 3 optimizer
  ablation should use per-optimizer learning rates rather than reusing this
  `lr0` for SGD.